# OTUS DZ07 — Multi-Agent Travel Assistant M1.1
Минимальный Colab-прототип показывает тот же архитектурный паттерн, что и основной код: `MessagesState`, типизированные LangChain messages, `Command` handoff и hybrid RAG (dense + lexical). Внешние API и секреты не нужны.

In [ ]:
%pip -q install 'langgraph>=0.2,<2' 'langchain-core>=0.3,<1'

In [ ]:
import hashlib, math, re
from typing import Any
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.types import Command

CHUNKS = [
    {'id': 'POLICY-FLIGHT-001', 'text': 'По России базовый класс перелёта — эконом.'},
    {'id': 'POLICY-HOTEL-001', 'text': 'Для Москвы гостиница допускается до 12 000 ₽ за ночь.'},
    {'id': 'POLICY-DAILY-001', 'text': 'Суточные по России — 1 500 ₽ в день.'},
]
ALIASES = {'отеля':'гостиница','отель':'гостиница','москве':'москва','москвы':'москва','перелета':'перелёт','перелет':'перелёт'}

def tokens(text):
    raw = re.findall(r'[a-zA-Zа-яА-ЯёЁ0-9]+', text.lower())
    return [ALIASES.get(x, x) for x in raw]

def embedding(text, dims=64):
    v = [0.0] * dims
    for token in tokens(text):
        d = hashlib.sha256(token.encode()).digest(); idx = int.from_bytes(d[:4], 'big') % dims
        v[idx] += 1.0 if d[4] % 2 == 0 else -1.0
    n = math.sqrt(sum(x*x for x in v)) or 1.0
    return [x/n for x in v]

def hybrid_search(query, top_k=2):
    qv = embedding(query); qs = set(tokens(query)); scored = []
    for c in CHUNKS:
        dense = max(0.0, sum(a*b for a,b in zip(qv, embedding(c['text']))))
        lexical = len(qs & set(tokens(c['text']))) / max(1, len(qs))
        scored.append({**c, 'dense': dense, 'lexical': lexical, 'score': .6*dense + .4*lexical})
    return sorted(scored, key=lambda x: x['score'], reverse=True)[:top_k]

class State(MessagesState):
    request: str
    evidence: list[dict[str, Any]]
    answer: str

def manager(state):
    return Command(goto='searcher', update={'messages':[AIMessage(content='Manager → Searcher: найди правила поездки', name='manager')]})

def searcher(state):
    hits = hybrid_search('перелёт гостиница Москва суточные', top_k=3)
    return Command(goto='finalize', update={'evidence':hits, 'messages':[AIMessage(content='PolicyRAG → Manager: правила найдены и ранжированы', name='policy_rag')]})

def finalize(state):
    refs = ', '.join(x['id'] for x in state['evidence'])
    answer = 'Эконом-перелёт; гостиница до 12 000 ₽/ночь; суточные 1 500 ₽/день. Evidence: ' + refs
    return {'answer': answer, 'messages':[AIMessage(content='Manager: итог сформирован', name='manager')]}

g = StateGraph(State)
g.add_node('manager', manager); g.add_node('searcher', searcher); g.add_node('finalize', finalize)
g.add_edge(START, 'manager'); g.add_edge('finalize', END)
app = g.compile()
result = app.invoke({'request':'Командировка Санкт-Петербург → Москва', 'evidence':[], 'answer':'', 'messages':[SystemMessage(content='Только read-only demo'), HumanMessage(content='Организуй командировку')]})
for m in result['messages']: print(type(m).__name__, getattr(m,'name',None), '→', m.content)
print('\nANSWER:', result['answer'])

## Production note
В полном варианте из `src/travel_multiagent_demo.py` добавлены dedup, reranking, chunk expansion, retrieval eval gate и специализированные Flight/Hotel/Budget agents.